In [0]:
# CONFIGURATION

STORAGE_ACCOUNT = "stlakehousedev225"
ACCESS_KEY       = "your_storage_account_key_here"
CATALOG          = "ecommerce_dev"
BRONZE_SCHEMA    = "bronze"
RAW_BASE         = f"abfss://raw@stlakehousedev225.dfs.core.windows.net/"
BRONZE_BASE      = f"abfss://bronze@stlakehousedev225.dfs.core.windows.net/"

In [0]:
# AUTHENTICATION

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    ACCESS_KEY
)
sc._jsc.hadoopConfiguration().set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    ACCESS_KEY
)
print("Authentication configured.")

Authentication configured.


In [0]:
# SET UNITY CATALOG CONTEXT

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")
print(f"Using catalog: {CATALOG}.{BRONZE_SCHEMA}")

Using catalog: ecommerce_dev.bronze


In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

files_to_tables = {
    "olist_orders_dataset.csv":              "orders",
    "olist_customers_dataset.csv":           "customers",
    "olist_order_items_dataset.csv":         "order_items",
    "olist_order_payments_dataset.csv":      "payments",
    "olist_order_reviews_dataset.csv":       "reviews",
    "olist_products_dataset.csv":            "products",
    "olist_sellers_dataset.csv":             "sellers",
    "olist_geolocation_dataset.csv":         "geolocation",
    "product_category_name_translation.csv": "category_translation",
}

for filename, table_name in files_to_tables.items():
    print(f"Ingesting {filename} → {CATALOG}.{BRONZE_SCHEMA}.{table_name} ...")
    
    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(f"{RAW_BASE}{filename}")
    
    df = df.withColumn("_ingest_ts", current_timestamp()) \
           .withColumn("_source_file", input_file_name())
    
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("path", f"{BRONZE_BASE}{table_name}") \
      .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    
    print(f"{table_name} — {df.count()} rows")

print("\nAll Bronze tables loaded.")

Ingesting olist_orders_dataset.csv → ecommerce_dev.bronze.orders ...
orders — 99441 rows
Ingesting olist_customers_dataset.csv → ecommerce_dev.bronze.customers ...
customers — 99441 rows
Ingesting olist_order_items_dataset.csv → ecommerce_dev.bronze.order_items ...
order_items — 112650 rows
Ingesting olist_order_payments_dataset.csv → ecommerce_dev.bronze.payments ...
payments — 103886 rows
Ingesting olist_order_reviews_dataset.csv → ecommerce_dev.bronze.reviews ...
reviews — 104162 rows
Ingesting olist_products_dataset.csv → ecommerce_dev.bronze.products ...
products — 32951 rows
Ingesting olist_sellers_dataset.csv → ecommerce_dev.bronze.sellers ...
sellers — 3095 rows
Ingesting olist_geolocation_dataset.csv → ecommerce_dev.bronze.geolocation ...
geolocation — 1000163 rows
Ingesting product_category_name_translation.csv → ecommerce_dev.bronze.category_translation ...
category_translation — 71 rows

All Bronze tables loaded.
